In [1]:
import pandas as pd
pd.set_option("display.max_colwidth",200)
import numpy as np
import re
import spacy
nlp=spacy.load("en_core_web_sm",disable=["parser","ner"])

In [2]:
df=pd.read_csv("tweets.csv", encoding="ISO-8859-1")

In [3]:
df.head()

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials to the experience... tacky.,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I need to take another trip!,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,"@VirginAmerica it's really aggressive to blast obnoxious ""entertainment"" in your guests' faces &amp; they have little recourse",NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing about it,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


In [4]:
print("shape=>",df.shape)


shape=> (14640, 15)


In [5]:
df['text'].sample(5)

12586                                                                                @AmericanAir Are you expecting any delays out of #DFW tomorrow morning?
1492     @united Flight 2 is 2:30 hrs delayed so far b/c of Late Flight crew. Now we are literally waiting while they have dinner acc. to honest gate agent.
12951                     @AmericanAir I paid using Paypal online and after I was charged there was a "System Error" which is what I ended up calling about.
12108                                            @AmericanAir delayed on the way to Puerto Rico and delayed on the way back to New York, this is disgraceful
4183                       @united flight 86 LAX-IAD, back rows NOT CLEANED prior to boarding. How gross is that to find used tissues in your seat?  Please.
Name: text, dtype: object

In [6]:
df['airline_sentiment'].value_counts()

airline_sentiment
negative    9178
neutral     3099
positive    2363
Name: count, dtype: int64

In [7]:
df['airline_sentiment'].value_counts(normalize=True)*100

airline_sentiment
negative    62.691257
neutral     21.168033
positive    16.140710
Name: proportion, dtype: float64

In [8]:
def text_cleaner(text):

    text = re.sub(r'@[A-Za-z0-9_]+', '', text)
    text = re.sub(r'#[A-Za-z0-9_]+', '', text)
    text = re.sub(r'http\S+', '', text)

    text = text.lower()

    # keep letters + spaces
    text = re.sub(r'[^a-z\s]', '', text)

    # remove extra spaces but keep one space
    text = re.sub(r'\s+', ' ', text).strip()

    doc = nlp(text)

    tokens = [
        token.lemma_
        for token in doc
        if not token.is_stop and not token.is_punct
    ]

    return " ".join(tokens)
    
    

In [9]:
df['clean_text']=df['text'].apply(text_cleaner)

In [10]:
text=df['clean_text'].values
labels=df['airline_sentiment'].values

In [11]:
text[:10]

array(['say', 'plus ve add commercial experience tacky',
       'not today mean need trip',
       'aggressive blast obnoxious entertainment guest face amp little recourse',
       'big bad thing',
       'seriously pay flight seat not play bad thing fly va',
       'yes nearly time fly vx ear worm will not away',
       'miss prime opportunity man hat parody', 'didntbut d',
       'amazing arrive hour early good'], dtype=object)

In [12]:
labels[:10]

array(['neutral', 'positive', 'neutral', 'negative', 'negative',
       'negative', 'positive', 'neutral', 'positive', 'positive'],
      dtype=object)

In [13]:
from sklearn.preprocessing import LabelEncoder

In [14]:
le=LabelEncoder()

In [15]:
labels=le.fit_transform(labels)

In [16]:
labels[:10]

array([1, 2, 1, 0, 0, 0, 2, 1, 2, 2])

In [17]:
le.inverse_transform([0,1,2])

array(['negative', 'neutral', 'positive'], dtype=object)

In [18]:
from sklearn.model_selection import train_test_split
x_train,x_val,y_train,y_val=train_test_split(text, labels, stratify=labels,test_size=0.2,random_state=0,shuffle=True)

In [19]:
print('x_train:', x_train.shape, 'y_train:',y_train.shape)
print('x_val:',x_val.shape, 'y_val:',y_val.shape)

x_train: (11712,) y_train: (11712,)
x_val: (2928,) y_val: (2928,)


In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [21]:
word_vectorizer=TfidfVectorizer(max_features=1000)

In [22]:
word_vectorizer.fit(x_train)

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,analyzer,'word'
,stop_words,None
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"


In [23]:
train_word_features=word_vectorizer.transform(x_train)
train_word_features

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 68961 stored elements and shape (11712, 1000)>

In [24]:
val_word_features=word_vectorizer.transform(x_val)
val_word_features

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 17427 stored elements and shape (2928, 1000)>

In [25]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score

In [26]:
nb_model=MultinomialNB().fit(train_word_features,y_train)
nb_model

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [27]:
train_pred_nb=nb_model.predict(train_word_features)

In [28]:
train_pred_nb

array([0, 0, 0, ..., 0, 0, 0], shape=(11712,))

In [29]:
print("F1-score on Train Set:",f1_score(y_train,train_pred_nb,average='weighted'))

F1-score on Train Set: 0.7287430397806316


In [30]:
val_pred_nb=nb_model.predict(val_word_features)

In [31]:
print("F1-score on Validation Set:",f1_score(y_val,val_pred_nb,average='weighted'))

F1-score on Validation Set: 0.6777359157570517


In [32]:
from sklearn.linear_model import LogisticRegression

In [33]:
lr_model=LogisticRegression().fit(train_word_features,y_train)
lr_model

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [34]:
train_pred_lr=lr_model.predict(train_word_features)
train_pred_lr

array([1, 0, 1, ..., 2, 0, 0], shape=(11712,))

In [35]:
print("F1-score on Train set:", f1_score(y_train,train_pred_lr,average="weighted"))

F1-score on Train set: 0.809258132131617


In [36]:
val_pred_lr=lr_model.predict(val_word_features)
val_pred_lr

array([0, 0, 2, ..., 0, 0, 0], shape=(2928,))

In [37]:
print("F1-score on Validation set:", f1_score(y_val,val_pred_lr,average="weighted"))

F1-score on Validation set: 0.7595523415836738


In [38]:
def sentiment_analyzer(tweet):
    cleaned_tweet=text_cleaner(tweet)
    tweet_vector=word_vectorizer.transform([cleaned_tweet])
    label=lr_model.predict(tweet_vector)
    return le.inverse_transform(np.array(label))

In [39]:
sentiment_analyzer("@united - That time when I spent a night trying to sleep in a toddler bed at the airport Ramada without my luggage you kidnapped #furious")

array(['negative'], dtype=object)

In [40]:
from sklearn.metrics import confusion_matrix, classification_report

print(classification_report(y_val, val_pred_lr))

print(confusion_matrix(y_val, val_pred_lr))

              precision    recall  f1-score   support

           0       0.81      0.92      0.86      1835
           1       0.62      0.51      0.56       620
           2       0.74      0.55      0.63       473

    accuracy                           0.77      2928
   macro avg       0.72      0.66      0.68      2928
weighted avg       0.76      0.77      0.76      2928

[[1680  111   44]
 [ 259  314   47]
 [ 131   81  261]]


In [41]:
print("LR F1:",
      f1_score(y_val, val_pred_lr, average="weighted"))

print("NB F1:",
      f1_score(y_val, val_pred_nb, average="weighted"))

LR F1: 0.7595523415836738
NB F1: 0.6777359157570517


In [42]:
import pickle

pickle.dump(lr_model, open("model.pkl", "wb"))
pickle.dump(word_vectorizer, open("vectorizer.pkl", "wb"))
pickle.dump(le, open("label_encoder.pkl", "wb"))